# Aquaplanet

This notebook builds a coupled aquaplanet directly in Python and runs it for
30 days: the SPEEDY T31L8 atmosphere over a slab ocean, with slab sea ice
diagnosed from the sea surface temperature. It then plots the atmosphere's
humidity, the ocean's temperature and an animation of the humidity. On a
laptop CPU it takes about 2 minutes, most of it compiling the coupled step
once.

It is JAX-ESM's own quick start -- the same construction
`docs/source/python_api.md` walks through, with the sea-ice component this
run also couples added on top (see the note below the build cell). The
closest command is

```bash
python -m jem.main +configuration=aquaplanet-slab
```

which has the same components, wiring and 30-minute atmosphere step, but not
the same atmosphere parameters: the configuration runs jax-gcm's SPEEDY
parameters calibrated for coupled use (its `speedy-t31` bundle), whereas a
plain `jcm.model.Model(...)`, as built here, runs SPEEDY's reference values.
This is a teaching build, not a calibrated or realistic climate; the
calibrated one is `jem.configurations.load("aquaplanet-slab")`.

Building it directly here, rather than composing that command's config and
calling `jem.runners.run(cfg)`, is the point of this notebook: every object
below is a plain Python class from `jem.components`, and nothing about it
depends on Hydra. The `Model(...)` call passes no `time_step`, so it takes
SPEEDY T31L8's stable step capped at 30 minutes, which is also the step the
configuration runs at; `tests/unit/test_notebook_equivalence.py` checks the
two build the same coupled structure, `dt` included.
`02_aquaplanet_customized_initial_condition.ipynb` builds these same
components and writes the exchange out as a plain function in place of
`default_exchangers`, so the coupling itself is on the page.
`03_aquaplanet_response_to_SST_perturbation_using_gradient.ipynb` instead uses
`jem.configurations.load("aquaplanet-slab")` -- the recipe *door* onto this
same validated configuration -- because what it demonstrates (differentiating
through a run) has nothing to do with how the model was assembled.

## Build it

In [ ]:
from pathlib import Path

from jem import plot

output_dir = (Path("output") / "01-01_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

This next cell is the direct construction: a plain `jcm.model.Model`
wrapped as a component, a `SlabGrid` taken from the atmosphere's own
horizontal grid, and a `Coupler` wiring them (plus the sea-ice component
below) with the standard exchange table. It is identical to
`docs/source/python_api.md`'s own quick-start block, up to and including
`print(repr(coupler))` -- pinned byte for byte by
`tests/unit/test_notebook_construction.py` -- except for the
import and the `components` entry `SlabSeaiceModel` needs.
`python_api.md`'s own block stops at the atmosphere and the ocean (the
smallest coupled pair worth showing); this run is `aquaplanet-slab`, which
also couples the slab sea-ice model, so it is added here to keep this
notebook honest about what it actually runs -- the ice fraction the
atmosphere's surface-flux scheme sees is then diagnosed from the coupled
ocean's sea surface temperature: full cover at the seawater freezing point,
none once the ocean is 1.8 K above it.

This aquaplanet is warm -- its sea surface temperature never gets near
freezing -- so the ice fraction is exactly zero everywhere, throughout the
run, and the sea-ice component changes nothing here. It is in the model
because it is part of `aquaplanet-slab`: it keeps the atmosphere's ice
boundary condition coupled to the ocean, so a colder experiment would
diagnose ice wherever the ocean cooled to within 1.8 K of freezing, with no
rewiring. The plots below therefore show the temperature margin to freezing
instead of an empty ice map.

In [ ]:
import jax_datetime as jdt
import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords

from jem import Coupler, default_exchangers, run_chunked
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid

start_date = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")

# The JCM atmosphere: a plain jcm.model.Model, wrapped as a component.
atm_model = jcm.model.Model(coords=get_speedy_coords(), start_time=start_date)
atm = JCMComponent(atm_model)

# Aquaplanet: the slab grid is built from the atmosphere's own horizontal grid,
# and with no fractional mask every cell is ocean.
grid = SlabGrid.from_coords(atm_model.coords.horizontal)

# An exchanger is the only place where components exchange information.
# `default_exchangers` is the standard wiring written down once — here, the
# atmosphere's surface heat flux drives the ocean and the ocean's SST comes
# back as the atmosphere's boundary condition — filtered to whichever of the
# standard components (`atm`, `ocn`, `lnd`, `seaice`) are present.
components = {
    "atm": atm,
    "ocn": SlabOceanModel(grid),
    "seaice": SlabSeaiceModel(grid, name="seaice"),
}
coupler = Coupler(
    components,
    default_exchangers(components),
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)
print(repr(coupler))

## Run it

In [ ]:
result = run_chunked(
    coupler,
    total_time="30 days",
    chunk="30 days",
    output_dir=str(output_dir),
    subsample=3,       # 10 records out of 30 coupled days
    checkpoint_path=None,
)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts on (`<component>-<first step>.nc`).

In [ ]:
atm_ds = plot.open_output(output_dir, "atm")
ocn_ds = plot.open_output(output_dir, "ocn")
seaice_ds = plot.open_output(output_dir, "seaice")
list(ocn_ds.data_vars)

## Plot

Three panels: the specific humidity of the atmosphere's lowest model level
(the "surface" humidity; animated below), the sea surface temperature of the
ocean, and the zonal-mean sea surface temperature against the seawater
freezing point -- the margin that keeps the ice fraction at zero (it is
printed below, rather than mapped, because it is zero everywhere).

In [ ]:
import matplotlib.pyplot as plt

import jem.constants as surface_constants

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# `level` is the atmosphere's sigma coordinate, ordered surface-first, so
# the level nearest sigma = 1 is the lowest model level (sigma = 0.95 in
# SPEEDY T31L8). Selecting by value rather than `.isel(level=0)` says so
# without relying on the ordering, and the title reads the level back.
humidity = atm_ds["specific_humidity"].sel(level=1.0, method="nearest")
humidity_title = (
    f"Specific humidity, lowest model level (sigma = {float(humidity.level):.2f})"
    " [kg/kg]"
)
plot.map_plot(humidity.isel(time=-1), ax=axes[0], title=humidity_title)

sst = ocn_ds["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[1], title="Sea surface temperature [°C]")

# Ice fraction is identically zero in this warm run (see the note under the
# build cell), so its map would be one flat colour; the quantity that
# explains it is how far the ocean stays above freezing.
freezing = surface_constants.seawater_freezing_point_K - 273.15
zonal_mean_sst = sst.mean("lon")
axes[2].plot(zonal_mean_sst["lat"], zonal_mean_sst, label="zonal-mean SST")
axes[2].axhline(freezing, color="tab:blue", linestyle="--",
                label=f"seawater freezing point ({freezing:.2f} °C)")
axes[2].set_xlabel("latitude [deg]")
axes[2].set_ylabel("[°C]")
axes[2].set_title("Zonal-mean SST against freezing")
axes[2].legend()
plt.tight_layout()

print("maximum sea ice fraction over the run:",
      float(seaice_ds["ice_fraction"].max()))
print("minimum sea surface temperature over the run [K]:",
      float(ocn_ds["sea_surface_temperature"].min()))

fig, ax = plt.subplots()
plot.area_mean(ocn_ds["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")

# Animate the same field the static panel above shows; each frame's title
# appends its own date, so the title is kept short enough to fit with it.
ani = plot.animate_map(
    humidity, title=f"Humidity, sigma = {float(humidity.level):.2f} [kg/kg]")
ani.save(output_dir / "specific_humidity.gif", writer="pillow")
# `animate_map` leaves its own figure current, which a notebook would otherwise also
# display as a stray static frame; the gif is its output.
plt.close(plt.gcf())
